# Ranking diagnostics: updated 120 development profiles

This analysis uses **D001-D120**, the current 378-occupation pool, and the matching weighted-dot-product and cosine `baseline_v1` evaluations. All six oracle dimensions are observed. Inputs are synthetic privileged `hidden_slots`, so these results measure ranking rather than extractor/tracker accuracy.

The previous version attempted to join the updated dataset to seven historical **60-profile** M4 evaluations. Their fingerprints differ and they cannot supply results for D061-D120. The local MiniLM cache also fails validation as stale. BM25, MiniLM, and fusion conclusions from those historical artifacts are therefore excluded here; each would require a matching new evaluation before comparison.

Run all cells with the project `.venv`. Analysis outputs are written separately under `reports/ranking/diagnostics_dev120/`. This notebook does not register diagnostic calculations in evaluation history or change ranking defaults.

New appendix sections compare four historical structured-only oracle runs and chart evaluation-history metrics over chronological experiments. Their results are sourced from the central CSV; the 120-profile diagnostic sections still use validated per-profile saved predictions.

In [ ]:
from pathlib import Path
import os, sys, json, hashlib
ROOT = Path.cwd().resolve()
if not (ROOT / 'data/dev_profiles.jsonl').is_file():
    ROOT = ROOT.parent
assert (ROOT / 'data/dev_profiles.jsonl').is_file()
os.chdir(ROOT)
if str(ROOT) not in sys.path: sys.path.insert(0, str(ROOT))
os.environ.setdefault('MPLCONFIGDIR', str(ROOT / '.venv/matplotlib-cache'))
%matplotlib inline
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
from recommender.data_loader import read_profiles, load_eligible_occupations
from recommender.evaluation import METRICS, aggregate, evaluate_ranking
from recommender.compare import paired_comparison
from recommender.schemas import TYPES
OUT = Path('reports/ranking/diagnostics_dev120')
OUT.mkdir(parents=True, exist_ok=True)
BASE = Path('reports/ranking/dot_vs_cosine_dev120')
RUNS = {'Weighted dot product': BASE / 'weighted_dot_product', 'Cosine similarity': BASE / 'cosine_similarity'}
def read(path): return json.loads(path.read_text(encoding='utf-8'))
def sha(path): return hashlib.sha256(path.read_bytes()).hexdigest()
protected = [Path('data/dev_profiles.jsonl'), Path('data/occupations.csv'), Path('reports/ranking/evaluation_history.csv'), Path('notebooks/ranking_evaluation_history_clean.csv')]
protected += [p for folder in RUNS.values() for p in folder.rglob('*') if p.is_file()]
before = {str(p): sha(p) for p in protected}
plt.rcParams.update({'figure.figsize': (9, 4), 'axes.spines.top': False, 'axes.spines.right': False})
pd.set_option('display.max_columns', 20)


## Validate the comparison

Require identical dataset and input fingerprints, profile IDs, eligibility, pool, ranking depth and metric definitions. Recompute every saved metric from full predictions. No exclusions or mixing of historical datasets are allowed.

In [ ]:
profiles = read_profiles(Path('data/dev_profiles.jsonl'))
assert [p['id'] for p in profiles] == [f'D{i:03}' for i in range(1, 121)]
pmap = {p['id']: p for p in profiles}
occupations = load_eligible_occupations(Path('data/occupations.csv'))
pool = {o.code for o in occupations}
assert len(pool) == 378
comparison, paired_rows = paired_comparison(RUNS['Weighted dot product'], RUNS['Cosine similarity'])
assert comparison['profiles'] == 120
manifests, rankings, metric_rows = {}, {}, []
history = pd.read_csv('reports/ranking/evaluation_history.csv')
for method, folder in RUNS.items():
    manifest = read(folder / 'manifest.json')
    manifests[method] = manifest
    for path, expected in manifest['dataset_sha256'].items(): assert sha(Path(path)) == expected, path
    assert sha(folder / 'inputs.json') == manifest['inputs_sha256']
    inputs = read(folder / 'inputs.json')
    assert len(inputs) == 120 and {r['profile_id'] for r in inputs} == set(pmap)
    for r in inputs:
        assert r['interests'] == pmap[r['profile_id']]['hidden_slots']
        assert r['user_sentences'] == [pmap[r['profile_id']]['text']]
    assert set(manifest['pool']) == pool
    assert (history.run_id == manifest['run_id']).sum() == 1
    by_id = {pid: [] for pid in pmap}
    for line in (folder / 'predictions.jsonl').read_text(encoding='utf-8').splitlines():
        r = json.loads(line)
        assert r['run_id'] == manifest['run_id']
        by_id[r['profile_id']].append(r)
    saved = {r['profile_id']: r for r in read(folder / 'per_profile.json')}
    recomputed = []
    for pid, items in by_id.items():
        items.sort(key=lambda r: r['rank'])
        codes = [r['code'] for r in items]
        assert len(codes) == len(set(codes)) == len(pool) and set(codes) == pool
        assert [r['rank'] for r in items] == list(range(1, len(pool)+1))
        p = pmap[pid]
        result = evaluate_ranking(codes, p['target'], p['related'], pool)
        for m in METRICS: assert np.isclose(result[m], saved[pid][m], atol=1e-12, rtol=0)
        recomputed.append(result)
        metric_rows.append({'profile_id': pid, 'method': method, **result})
    summary = read(folder / 'metrics.json')
    for m in METRICS: assert np.isclose(aggregate(recomputed)[m], summary[m], atol=1e-12, rtol=0)
    rankings[method] = by_id
metrics = pd.DataFrame(metric_rows)
display(pd.DataFrame([{'method': m, 'profiles': 120, 'candidates': len(pool), 'mode': manifests[m]['input_mode'], 'run_id': manifests[m]['run_id']} for m in RUNS]))


## Overall ranking results

Hit@1/5/10 requires the exact assigned target. MRR uses the full candidate ranking. NDCG@5/10 uses unchanged evaluator grades: target 4, Primary-Short 3, Primary-Long 2, Supplemental 1. Confidence intervals below use 2,000 paired profile bootstrap samples, seed 425, for cosine minus dot product.

In [ ]:
overall = metrics.groupby('method')[list(METRICS)].mean().reindex(RUNS)
display(overall.round(6))
ci = pd.DataFrame(comparison['metrics']).T
display(ci)
overall[['hit_at_5', 'mrr', 'ndcg_at_10']].plot.bar(rot=0, ylim=(0, 1))
plt.title('Same 120 oracle profiles: overall ranking quality'); plt.ylabel('Metric value'); plt.tight_layout(); plt.show()
overall.to_csv(OUT / 'overall_metrics.csv')
ci.to_csv(OUT / 'paired_comparison.csv')


## Historical four-run comparison: scoring method × dataset size

This section reads `reports/ranking/evaluation_history.csv` and selects **the latest structured-only oracle run** for each of these four cells: weighted dot product (60), cosine (60), weighted dot product (120), cosine (120). Historical and latest experiments are reported without rewriting source artifacts. Comparisons across 60 and 120 profiles are **not controlled paired algorithm comparisons**, because the evaluation population changes; within each size the two scoring methods are compared under the same dataset fingerprint and candidate pool. The notebook uses the current 120-profile per-profile diagnostic tables separately below.

In [ ]:
# Reusable helpers for the four historical structured-only comparisons.
from matplotlib.ticker import PercentFormatter
from matplotlib.lines import Line2D

HISTORY_PATH = ROOT / 'reports/ranking/evaluation_history.csv'
assert HISTORY_PATH.is_file(), f'Missing evaluation history: {HISTORY_PATH}'
HISTORY_METRICS = ('hit_at_1', 'hit_at_5', 'hit_at_10', 'mrr', 'ndcg_at_5', 'ndcg_at_10')


def load_evaluation_history(path=HISTORY_PATH):
    df = pd.read_csv(path, dtype={'run_id': str, 'dataset_fingerprint': str})
    required = {'run_id', 'timestamp', 'm1', 'm3', 'm4', 'fusion', 'profile_count',
                'candidate_count', 'dataset_fingerprint', *HISTORY_METRICS}
    missing = required - set(df.columns)
    if missing:
        raise ValueError(f'Evaluation history is missing: {sorted(missing)}')
    df['timestamp'] = pd.to_datetime(df['timestamp'], utc=True, errors='coerce')
    if df.timestamp.isna().any():
        raise ValueError('Some experiment timestamps cannot be parsed')
    for name in ['m1', 'm3', 'm4', 'fusion']:
        df[name] = df[name].fillna('').astype(str).str.strip()
    for name in ['profile_count', 'candidate_count', *HISTORY_METRICS]:
        df[name] = pd.to_numeric(df[name], errors='coerce')
    df['structured_method'] = np.select(
        [df.m1.str.startswith('weighted_dot_product'), df.m1.str.startswith('cosine_similarity')],
        ['Dot product', 'Cosine'], default='Other/none')
    df['components'] = df.apply(lambda r: ' + '.join(
        [r.structured_method if r.structured_method != 'Other/none' else ('M1' if r.m1 else ''),
         'BM25' if r.m3 else '', 'MiniLM' if r.m4 else '']).strip(' +') or 'None', axis=1)
    df['experiment_label'] = df['components'] + ' · n=' + df.profile_count.astype('Int64').astype(str)
    df['input_mode_group'] = np.select([
        df.run_id.str.startswith('m2_structured_oracle'),
        df.run_id.str.startswith('m5_mock_tracker'),
        df.run_id.str.startswith('m6_stress')],
        ['oracle', 'mock_tracker', 'mock_tracker'], default='other')
    # Stable sort if timestamps coincide; chronological not a progression-controlled comparison.
    return df.sort_values(['timestamp', 'run_id'], kind='stable').reset_index(drop=True)


def select_four_structured_runs(history):
    eligible = history[
        history.input_mode_group.eq('oracle') &
        history.structured_method.isin(['Dot product', 'Cosine']) &
        history.profile_count.isin([60, 120]) &
        history.m3.eq('') & history.m4.eq('') & history.fusion.eq('')
    ].copy()
    # Among duplicate historical runs, take the latest for each method and size.
    chosen = eligible.sort_values(['timestamp', 'run_id']).drop_duplicates(
        ['profile_count', 'structured_method'], keep='last')
    expected = {(n, m) for n in (60, 120) for m in ('Dot product', 'Cosine')}
    actual = set(zip(chosen.profile_count.astype(int), chosen.structured_method))
    if actual != expected:
        raise ValueError(f'Missing structured-only runs: {sorted(expected - actual)}')
    # Check same candidate pool + fingerprint within each dataset size.
    for count, part in chosen.groupby('profile_count'):
        if part.candidate_count.nunique() != 1 or part.dataset_fingerprint.nunique() != 1:
            raise ValueError(f'{count}-profile structured runs have incompatible pool or dataset fingerprints')
    return chosen.sort_values(['profile_count', 'structured_method'], ascending=[True, False]).reset_index(drop=True)


def four_model_table(runs, metrics=HISTORY_METRICS):
    display(runs[['structured_method', 'profile_count', 'candidate_count', *metrics,
                  'timestamp', 'dataset_fingerprint', 'run_id']].style.format(
        {metric: '{:.2%}' for metric in metrics}))


def plot_four_model_metrics(runs, metrics=HISTORY_METRICS):
    fig, axes = plt.subplots(2, 3, figsize=(15, 8), sharey=True)
    order = [('Dot product', 60), ('Cosine', 60), ('Dot product', 120), ('Cosine', 120)]
    labels = ['Dot · 60', 'Cos · 60', 'Dot · 120', 'Cos · 120']
    lookup = {(r.structured_method, int(r.profile_count)): r for r in runs.itertuples()}
    for ax, metric in zip(axes.flat, metrics):
        heights = [getattr(lookup[k], metric) for k in order]
        ax.bar(range(4), heights)
        ax.set_title(metric.upper()); ax.set_xticks(range(4), labels, rotation=25, ha='right')
        ax.set_ylim(0, 1); ax.yaxis.set_major_formatter(PercentFormatter(1.0))
    fig.suptitle('Structured-only oracle experiments: 60 vs 120 profiles')
    fig.tight_layout()
    plt.show()
    return fig


def compare_method_and_dataset(runs, metric='hit_at_5'):
    tab = runs.pivot(index='profile_count', columns='structured_method', values=metric).sort_index()
    tab['Cosine − Dot (pp)'] = (tab['Cosine'] - tab['Dot product']) * 100
    display(tab.style.format({'Cosine': '{:.2%}', 'Dot product': '{:.2%}', 'Cosine − Dot (pp)': '{:+.2f}'}))
    return tab

history_all = load_evaluation_history()
four = select_four_structured_runs(history_all)
four_model_table(four)
plot_four_model_metrics(four)
compare_method_and_dataset(four, 'hit_at_5')


## Updated dataset composition

D001-D060 retain their original profile content and add metadata. D061-D120 broaden top-type coverage and reshape interests into six patterns. `top_type` uses fixed R,I,A,S,E,C order to break ties. Balance of top types does not imply balance of patterns, tiers or occupation targets.

In [ ]:
vectors = pd.DataFrame([p['hidden_slots'] for p in profiles], index=[p['id'] for p in profiles])[list(TYPES)]
meta = pd.DataFrame([{'profile_id': p['id'], 'cohort': 'Original D001-D060' if int(p['id'][1:]) <= 60 else 'Added D061-D120', 'top_type': p['top_type'], 'pattern': p['pattern'], 'tier': p['tier'], 'style': p['style'], 'target': p['target']} for p in profiles])
assert all(vectors.loc[r.profile_id].idxmax() == r.top_type for r in meta.itertuples())
display(pd.crosstab(meta.top_type, meta.cohort).reindex(TYPES))
display(pd.crosstab(meta.pattern, meta.cohort))
display(pd.crosstab(meta.tier, meta.cohort))
display(vectors.describe().T.round(3))
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
pd.crosstab(meta.top_type, meta.cohort).reindex(TYPES).plot.bar(ax=axes[0], rot=0)
pd.crosstab(meta.pattern, meta.cohort).plot.barh(ax=axes[1])
axes[0].set_title('Top-type coverage'); axes[1].set_title('Interest patterns'); plt.tight_layout(); plt.show()


## Performance by cohort, top type, pattern and tier

These are descriptive subgroup comparisons. Cohorts differ in construction and type mix, so a cohort difference cannot be attributed solely to sample size. Small pattern groups make estimates uncertain. All subgroup counts below are shown explicitly.

In [ ]:
long = metrics.merge(meta, on='profile_id', validate='many_to_one')
subgroups = []
for attribute in ['cohort', 'top_type', 'pattern', 'tier', 'style']:
    table = long.groupby([attribute, 'method'], observed=True).agg(n=('profile_id', 'size'), hit_at_5=('hit_at_5', 'mean'), mrr=('mrr', 'mean'), ndcg_at_10=('ndcg_at_10', 'mean'))
    display(Markdown(f'**By {attribute}**')); display(table.round(4))
    subgroups.append(table.reset_index().rename(columns={attribute: 'group'}).assign(attribute=attribute))
pd.concat(subgroups, ignore_index=True).to_csv(OUT / 'subgroup_metrics.csv', index=False)
long.groupby(['pattern', 'method']).hit_at_5.mean().unstack().plot.barh(xlim=(0, 1))
plt.title('Exact-target Hit@5 by interest pattern'); plt.tight_layout(); plt.show()


## Assigned-target accuracy versus numerical fit

New profiles can be intentionally far from the target occupation average. Alongside unchanged official metrics, report two **diagnostic** hit rates: target or a Primary-Short related occupation in the top five; and target or one of the supplied `nearest_by_profile` occupations in the top five. Profile-fit labels are cosine-derived, so this measure structurally favors cosine and is not independent evidence of career suitability. These diagnostics are not added to the central history schema.

In [ ]:
fit_rows = []
for method, by_id in rankings.items():
    for pid, items in by_id.items():
        p = pmap[pid]
        top5 = {r['code'] for r in items[:5]}
        short = p['related'].get('primary_short', p['related'].get('short', []))
        fit_rows.append({'profile_id': pid, 'method': method, 'related_hit_at_5': bool(top5 & ({p['target']} | set(short))), 'fit_hit_at_5': bool(top5 & ({p['target']} | set(p['nearest_by_profile'])))})
fit = pd.DataFrame(fit_rows).merge(meta, on='profile_id', validate='many_to_one')
display(fit.groupby('method')[['related_hit_at_5', 'fit_hit_at_5']].mean().round(4))
display(fit.groupby(['pattern', 'method']).agg(n=('profile_id', 'size'), related_hit_at_5=('related_hit_at_5', 'mean'), fit_hit_at_5=('fit_hit_at_5', 'mean')).round(4))
fit.to_csv(OUT / 'fit_diagnostics.csv', index=False)


## Numerical alignment and individual regressions

Inspect user-to-target cosine and distance separately from algorithm quality. The weighted dot product rewards magnitude as well as direction; cosine normalizes both vectors. Review cases where cosine loses as well as where it wins.

In [ ]:
occ = {o.code: o for o in occupations}
alignment = []
for p in profiles:
    u = np.array([p['hidden_slots'][t] for t in TYPES])
    z = np.array([occ[p['target']].z_scores[t] for t in TYPES])
    alignment.append({'profile_id': p['id'], 'user_target_cosine': u @ z / (np.linalg.norm(u)*np.linalg.norm(z)), 'user_target_l2': np.linalg.norm(u-z)})
ranks = metrics.pivot(index='profile_id', columns='method', values='target_rank')
ranks['cosine_rank_improvement'] = ranks['Weighted dot product'] - ranks['Cosine similarity']
cases = meta.merge(ranks.reset_index(), on='profile_id').merge(pd.DataFrame(alignment), on='profile_id')
cases['text'] = cases.profile_id.map({p['id']: p['text'] for p in profiles})
display(cases.groupby('cohort')[['user_target_cosine', 'user_target_l2']].agg(['mean', 'median']).round(3))
display(Markdown('**Largest cosine regressions (negative rank improvement)**'))
display(cases.sort_values('cosine_rank_improvement').head(8))
display(Markdown('**Largest cosine improvements**'))
display(cases.sort_values('cosine_rank_improvement', ascending=False).head(8))
plt.scatter(cases['Weighted dot product'], cases['Cosine similarity'], alpha=.6)
plt.plot([1, 378], [1, 378], '--', color='gray'); plt.xscale('log'); plt.yscale('log')
plt.xlabel('Dot-product target rank'); plt.ylabel('Cosine target rank'); plt.title('Below diagonal: cosine ranks the target higher'); plt.tight_layout(); plt.show()
cases.to_csv(OUT / 'profile_diagnostics.csv', index=False)
long.to_csv(OUT / 'per_profile_metrics.csv', index=False)


## Findings and limits

The original 60-profile artifacts remain historical evidence only. Current results should be compared within this 120-profile cohort. Balance and pattern analyses explain composition; they do not establish causal effects. No held-out test profiles are read. Production algorithm selection requires separate tracker and dialogue validation.

**Cohort finding:** Cosine Hit@5 is 88.33% on D001-D060 and 48.33% on D061-D120; weighted dot product is 16.67% on each. The added cohort is more difficult under the exact assigned-target measure, while also differing in construction and type mix. This is a descriptive association, not a causal estimate.

**Fit-label caveat:** Cosine achieves 100% diagnostic fit-hit@5 because the supplied nearest occupations are themselves selected using cosine. This is useful for checking agreement with the dataset construction, but cannot independently validate cosine or career suitability.

In [ ]:
dot, cos = overall.loc['Weighted dot product'], overall.loc['Cosine similarity']
display(Markdown(f"On **{len(profiles)} identical development/oracle profiles**, cosine Hit@5 is **{cos.hit_at_5:.2%}** versus **{dot.hit_at_5:.2%}** for weighted dot product. MRR is **{cos.mrr:.4f}** versus **{dot.mrr:.4f}**; NDCG@10 is **{cos.ndcg_at_10:.4f}** versus **{dot.ndcg_at_10:.4f}**. These are synthetic ranking results, not measured student outcomes."))
assert {str(p): sha(p) for p in protected} == before, 'Protected inputs or experiment artifacts changed'
(OUT / 'analysis_manifest.json').write_text(json.dumps({'profile_count': len(profiles), 'run_ids': {m: v['run_id'] for m,v in manifests.items()}, 'protected_sha256': before, 'history_rows_added_by_notebook': 0}, indent=2), encoding='utf-8')
display(Markdown('**Integrity verified:** source datasets, saved evaluations and both history files remain unchanged. Diagnostic exports are in `reports/ranking/diagnostics_dev120/`.'))


## Chronological evaluation history: metric fluctuations across experiments

**X-axis = recorded experiment sequence** (sorted by UTC timestamp, oldest first). Each point represents one recorded run, not a training epoch. Six metric lines are shown on a common 0–1 scale. Use `mode='oracle'` to compare the oracle evaluation experiments only; `mode='all'` displays every run, visually separating oracle from mock-tracker runs. Changes in profile count, dataset fingerprint, fusion configuration and input mode can cause jumps: **do not interpret this figure as a learning curve or evidence of monotonic improvement.**

In [ ]:
def plot_metric_history(history, mode='oracle', metrics=HISTORY_METRICS,
                        candidate_count=None, figsize=(17, 7), annotate=True):
    """Line chart of experiment history, ordered by time. Returns the plotted rows."""
    data = history.copy()
    if mode != 'all':
        data = data.loc[data.input_mode_group.eq(mode)].copy()
    if candidate_count is not None:
        data = data.loc[data.candidate_count.eq(candidate_count)].copy()
    if data.empty:
        raise ValueError('No history rows match the requested filters')
    data = data.sort_values(['timestamp', 'run_id'], kind='stable').reset_index(drop=True)
    data['iteration'] = np.arange(1, len(data) + 1)
    fig, ax = plt.subplots(figsize=figsize)
    for metric in metrics:
        ax.plot(data['iteration'], data[metric], marker='o', markersize=3.5,
                linewidth=1.5, label=metric)
    ax.set(xlabel='Experiment sequence (chronological run index)',
           ylabel='Metric value', title=f'Evaluation history · {mode} · {len(data)} runs',
           ylim=(-.025, 1.025))
    ax.yaxis.set_major_formatter(PercentFormatter(1.0))
    ax.set_xticks(data['iteration'])
    ax.tick_params(axis='x', labelsize=8)
    ax.grid(True, alpha=.2)
    # Mark transitions across dataset fingerprints and modes.
    for i in range(1, len(data)):
        prev, curr = data.iloc[i-1], data.iloc[i]
        if (prev.dataset_fingerprint != curr.dataset_fingerprint or
            prev.input_mode_group != curr.input_mode_group):
            ax.axvline(i + .5, color='gray', linestyle=':', linewidth=1.2, alpha=.8)
    if annotate:
        for _, row in data.iterrows():
            ax.annotate(str(int(row.profile_count)), (row.iteration, .015),
                        xytext=(0, 1), textcoords='offset points', ha='center',
                        fontsize=7, color='gray')
    ax.legend(loc='upper left', bbox_to_anchor=(1.01, 1), title='Evaluation metrics')
    fig.tight_layout()
    plt.show()
    display(data[['iteration', 'timestamp', 'experiment_label', 'input_mode_group',
                  'fusion', *metrics, 'run_id']].round(4))
    return data

# Main plot: compare all six metrics across every oracle experiment in time order.
oracle_timeline = plot_metric_history(history_all, mode='oracle')

# Optional: reveal all recorded evaluations (including synthetic mock-tracker M5/M6).
# all_timeline = plot_metric_history(history_all, mode='all')

# Optional: break out cohort-specific trajectories to prevent misleading joins.
# oracle_60 = plot_metric_history(history_all[history_all.profile_count.eq(60)], mode='oracle')
# oracle_120 = plot_metric_history(history_all[history_all.profile_count.eq(120)], mode='oracle')
